# AgroAgent — LLM su Colab (Qwen3.6-35B-A3B GGUF via llama.cpp)

Serve `unsloth/Qwen3.6-35B-A3B` (MoE, 35B totali / ~3B attivi) quantizzato **UD-Q4_K_XL** (~20 GB) con `llama-server`, ed espone un endpoint **OpenAI-compatible** all'esterno tramite un tunnel Cloudflare.

**Requisiti:** Colab Pro con GPU **A100 (40 GB)**. Runtime → *Change runtime type* → A100.

**Come si usa:** esegui le celle dall'alto verso il basso. L'ultima cella stampa le 3 righe da incollare nel `.env` dello stack AgroAgent. L'URL del tunnel **cambia a ogni sessione**: a ogni riavvio riesegui il notebook e aggiorna `OPENAI_BASE_URL` nel `.env`.

## 1. Verifica GPU
Deve comparire una **A100** con ~40 GB. Se vedi una T4/L4, cambia il tipo di runtime.

In [ ]:
!nvidia-smi

## 2. Build di llama.cpp con CUDA
Compila `llama-server` con supporto GPU. Richiede ~3-5 minuti. `-DLLAMA_CURL=ON` abilita il download diretto del modello da HuggingFace con `-hf`.

> Qwen3.6-35B-A3B è recente: si clona `main` per avere il supporto all'architettura e al chat template (tool calling).

In [ ]:
!apt-get -qq update && apt-get -qq install -y libcurl4-openssl-dev
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cmake llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON
!cmake --build llama.cpp/build --config Release -j --target llama-server

## 3. API key
L'endpoint via tunnel è pubblico (raggiungibile da chiunque conosca l'URL), quindi l'api-key **non è opzionale**. Questo stesso valore andrà in `OPENAI_API_KEY` nel `.env`.

In [ ]:
import os, secrets

LLAMA_API_KEY = secrets.token_hex(16)
os.environ["LLAMA_API_KEY"] = LLAMA_API_KEY
print("API key (la riuserai nel .env come OPENAI_API_KEY):", LLAMA_API_KEY)

## 4. Avvio di llama-server (in background)
Avvia il server come processo figlio del kernel, così sopravvive tra le celle. Note sui flag:
- `-hf unsloth/Qwen3.6-35B-A3B-GGUF:UD-Q4_K_XL` — scarica il quant scelto da HuggingFace (la **prima volta** sono ~20 GB: richiede alcuni minuti).
- `--jinja` — **obbligatorio** per il tool calling (usa il chat template del modello per emettere/parsare i tool call). Senza, l'agente AgroAgent non invocherà mai i tool MCP.
- `--alias qwen3.6-35b-a3b` — nome stabile del modello; è il valore che metterai in `LLM_MODEL`.
- `-ngl 99` — tutti i layer su GPU. `-c 32768` — context (la KV cache di un MoE dipende dai layer di attenzione, non dagli expert → 32k entra comodo nei 40 GB).

In [ ]:
import subprocess

log = open("llama_server.log", "w")
server = subprocess.Popen(
    [
        "./llama.cpp/build/bin/llama-server",
        "-hf", "unsloth/Qwen3.6-35B-A3B-GGUF:UD-Q4_K_XL",
        "--alias", "qwen3.6-35b-a3b",
        "--jinja",
        "-ngl", "99",
        "-c", "32768",
        "--host", "0.0.0.0", "--port", "8000",
        "--api-key", os.environ["LLAMA_API_KEY"],
    ],
    stdout=log, stderr=subprocess.STDOUT,
)
print("llama-server avviato, PID:", server.pid)

## 5. Attesa che il server sia pronto
La prima volta scarica il modello (~20 GB) e poi lo carica in VRAM: può richiedere diversi minuti. Il loop attende l'endpoint `/health`.

In [ ]:
import time, requests

ready = False
for i in range(240):  # ~20 min max (download + load)
    try:
        if requests.get("http://localhost:8000/health", timeout=5).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    if i % 6 == 0:
        print(f"...in attesa ({i * 5}s)")
    time.sleep(5)

if ready:
    print("Server pronto.")
else:
    print("Timeout. Ultime righe di log:")
    print(open("llama_server.log").read()[-3000:])

## 6. Tunnel HTTPS (Cloudflare quick tunnel)
Espone `localhost:8000` su un URL pubblico `https://<...>.trycloudflare.com`. Nessun account richiesto. L'URL **cambia a ogni avvio**.

In [ ]:
import subprocess, re, time

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(40):
    time.sleep(2)
    try:
        text = open("cloudflared.log").read()
    except FileNotFoundError:
        text = ""
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", text)
    if m:
        public_url = m.group(0)
        break

print("Tunnel URL:", public_url or "NON trovato — controlla cloudflared.log")

## 7. Smoke test del tool calling
Verifica **end-to-end** (attraverso il tunnel) che il modello produca un `tool_calls` strutturato. L'agente AgroAgent dipende da questo: se qui non compare, non funzionerà nemmeno lo stack.

In [ ]:
import requests, json

resp = requests.post(
    f"{public_url}/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['LLAMA_API_KEY']}"},
    json={
        "model": "qwen3.6-35b-a3b",
        "messages": [
            {"role": "user", "content": "Che tempo fa a Roma? Usa lo strumento disponibile."}
        ],
        "tools": [
            {
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "description": "Ottiene il meteo attuale per una citta",
                    "parameters": {
                        "type": "object",
                        "properties": {"city": {"type": "string"}},
                        "required": ["city"],
                    },
                },
            }
        ],
        "tool_choice": "auto",
    },
    timeout=180,
)
data = resp.json()
print(json.dumps(data, indent=2, ensure_ascii=False))

tool_calls = data.get("choices", [{}])[0].get("message", {}).get("tool_calls")
print("\n==> TOOL CALLING OK" if tool_calls else "\n==> NESSUN tool_call: controlla che --jinja sia attivo e la versione di llama.cpp aggiornata")

## 8. Righe per il `.env` di AgroAgent
Copia queste righe nel `.env` sul tuo host, poi riavvia solo l'Agent Service:

```bash
docker compose up -d agent_service
```

In [ ]:
print("# --- incolla nel .env ---")
print(f"OPENAI_BASE_URL={public_url}/v1")
print(f"OPENAI_API_KEY={os.environ['LLAMA_API_KEY']}")
print("LLM_MODEL=qwen3.6-35b-a3b")
print("\n# poi sul tuo host: docker compose up -d agent_service")

## Note operative
- **URL effimero:** il tunnel cambia a ogni sessione Colab. Al riavvio riesegui il notebook e aggiorna `OPENAI_BASE_URL` nel `.env`, poi `docker compose up -d agent_service`.
- **`LLM_MODEL` deve combaciare con `--alias`** (`qwen3.6-35b-a3b`): è il campo `model` che l'Agent Service invia.
- **Sessioni a tempo:** Colab stacca il runtime per inattività / dopo molte ore. Adatto a demo e test, non a un servizio "sempre acceso".
- **Sicurezza:** non committare mai l'URL del tunnel né l'api-key. L'api-key resta solo nel `.env` (mai esposta al frontend).
- **Reasoning di Qwen3.6:** l'eventuale blocco `<think>...</think>` viene gestito dall'Agent Service, che lo rimuove dalla risposta finale — nessuna azione richiesta.
- **Quant alternative** (cambia il tag in `-hf ...:TAG`): `UD-Q5_K_XL`/`UD-Q6_K_XL` per più qualità se avanza VRAM; `UD-Q3_K_XL` per più margine. Default consigliato a 40 GB: `UD-Q4_K_XL`.
- **Diagnostica:** log del modello in `llama_server.log`, log del tunnel in `cloudflared.log`.